# The campaign threshold Λ_c: the closed form tested in ten cases (Sect. 5.1)

**In the paper:** Sect. 5.1 (the threshold and its analytic approximation, Vitells & Gross 2011; "≲ 1 %").

`thr.lrt_threshold` is a formula from the literature: the expected Euler characteristic of a Gaussian field,

$$P(\max Z > t) \approx \sum_d L_d\,\rho_d(t),\qquad
\rho_d(t) = (2\pi)^{-(d+1)/2}\,\mathrm{He}_{d-1}(t)\,e^{-t^2/2},$$

with the curvatures $L_d$ of the searched region computed from the instrument and the scan law
(`thr.search_curvatures`). [`sim_lambda_c.ipynb`](sim_lambda_c.ipynb) checks it for the paper's case, Figure 4's.
This notebook checks it in **ten cases**: the paper's, and nine that change one thing each (the scan law, the
searched region, the instrument, a search for sources at rest). In each case the formula's level is compared
with the level of simulated noise-only campaigns (`sim.field_null_max`), at false rates of 10 % and 1 %.

**What is simulated.** `sim.field_null_max` draws noise in every window and returns the maximum over the
searched region of $Z$, the candidate's best-fit flux over its error: the Gaussian field of the theory. It
does not run the joint fits of the search. In every case the noise is the same on every sample, the primary
is pinned, and the zero points are 0: the assumptions of the formula.

| step | task | calls | paper |
|---|---|---|---|
| 1 | the ten cases | `scanlaw.load_scan_law`, `ins.Instrument.gaussian` | Sect. 5.1 |
| 2 | the formula in each case | `thr.lrt_threshold`, `thr.search_curvatures` | Sect. 5.1 |
| 3 | the simulated null of each case | `sim.field_null_max` | Sect. 5.1 |
| 4 | the false rate at the formula's levels, and the simulated levels | — | Sect. 5.1 |
| 5 | the figures: the distributions, the rates, the survival curves | — | — |

Every scanlib call carries the name of its module (`thr.lrt_threshold`); the first code cell lists the modules.

**Run time.** The 16 000 simulated campaigns take about 20 hours on one core (1 to 20 s each), so they are stored
in `../stored/lambda_c_validation.npz`. By default the notebook loads them and recomputes the first campaign of
each case, which must equal the stored one bit for bit. About 1 min. `FULL = True` recomputes all of them and
rewrites the store.

In [ ]:
# the imports: numpy, matplotlib, scipy's beta distribution, and the scanlib modules; every scanlib call below
# carries its module's name
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta

from scanlib import instrument as ins      # the instrument: sampling, LSF, the along-scan projection (Eq. 9)
from scanlib import scanlaw                # scan angles, epochs and parallax factors
from scanlib import simulation as sim      # synthetic scenes and their windows; the search's null field
from scanlib import thresholds as thr      # the threshold of the search, Lambda_c

In [ ]:
# the switch FULL and the store of the simulated null
FULL = False                               # True: recompute the 16 000 campaigns (about 20 h on one core)
STORE = Path("..") / "stored" / "lambda_c_validation.npz"

## 1. The ten cases (Sect. 5.1)

The **reference** is the paper's search: the 47 transits of the Pleiades law, the field of ±600 mas minus the
30 mas zone of the primary, proper motions within ±10 mas/yr, a Gaussian LSF of FWHM 100 mas. Each other case
changes one entry.

In [ ]:
# the two scan laws of the package: 47 transits (the Pleiades centre) and 95 transits (ecliptic latitude +60 deg)
# psi_*: the scan angle of each transit [rad]; tau_*: its epoch [yr from the reference epoch]
psi_47, tau_47 = scanlaw.load_scan_law(scanlaw.PLEIADES)
psi_95, tau_95 = scanlaw.load_scan_law(scanlaw.BETA60)

# np.ptp: the largest minus the smallest value, here the time that the law covers
print(f"Pleiades: {psi_47.size} transits over {np.ptp(tau_47):.1f} yr;  "
      f"latitude +60: {psi_95.size} transits over {np.ptp(tau_95):.1f} yr")

In [ ]:
# the cases: the reference, and nine copies of it with one entry changed (dict(REFERENCE, key=value) is such a copy);
# psi_47[::3] keeps every third transit, 16 of the 47
# REFERENCE: the table of the settings of the paper's search, with six entries
#     "psi", "tau"   the scan law: the scan angle [rad] and the epoch [yr] of each transit
#     "static"       False: the search is for moving sources; True: for sources at rest
#     "half_size"    the half-width of the searched field [mas]
#     "mu_max"       the largest searched proper motion, per component [mas/yr]
#     "fwhm"         the FWHM of the Gaussian LSF [mas]
REFERENCE = dict(psi=psi_47, tau=tau_47, static=False, half_size=600.0, mu_max=10.0, fwhm=100.0)
# CASES: a table from a case's name to the table of its settings; CASES["field ±300 mas"]["half_size"] is 300
CASES = {
    "reference": REFERENCE,
    "95 transits": dict(REFERENCE, psi=psi_95, tau=tau_95),
    "16 transits": dict(REFERENCE, psi=psi_47[::3], tau=tau_47[::3]),
    "at rest, 47 transits": dict(REFERENCE, static=True),
    "at rest, 95 transits": dict(REFERENCE, psi=psi_95, tau=tau_95, static=True),
    "field ±300 mas": dict(REFERENCE, half_size=300.0),
    "motions ±5 mas/yr": dict(REFERENCE, mu_max=5.0),
    "motions ±20 mas/yr": dict(REFERENCE, mu_max=20.0),
    "LSF FWHM 70 mas": dict(REFERENCE, fwhm=70.0),
    "LSF FWHM 150 mas": dict(REFERENCE, fwhm=150.0),
}
NAMES = tuple(CASES)                       # the ten names, in the order above; a case's number is its place here

print(f"{len(CASES)} cases:", ", ".join(NAMES))

In [ ]:
# the instrument of a case: the toy window (18 samples of 59 mas) with a Gaussian LSF of the case's FWHM
def instrument(case):
    """The instrument of a case; only the LSF's FWHM differs between cases.

    Inputs
        case   the case's name, a key of CASES
    Returns an ins.Instrument: the toy window (18 samples of 59 mas), a Gaussian LSF of the case's FWHM [mas], and
    the toy's per-window margins."""
    return ins.Instrument.gaussian(fwhm_mas=CASES[case]["fwhm"], pitch_mas=59.0, n_samples=18, margins=ins.TOY_MARGINS)

## 2. The formula in each case (Sect. 5.1)

`thr.lrt_threshold` returns Λ_c = t², with t the solution of Σ_d L_d ρ_d(t) = p. A search for sources at rest
has no epochs (`tau` is not given): its region has two dimensions, the sky position. A moving search has four.

In [ ]:
# the formula's level of a case at a false rate p
def formula(case, false_rate):
    """Lambda_c of a case from the closed form (thr.lrt_threshold).

    Inputs
        case         the case's name, a key of CASES
        false_rate   the fraction of noise-only campaigns that may exceed the level (0.01 is 1 %)
    Returns Lambda_c, the level of Lambda = max(Z, 0)^2 that the formula gives for that false rate."""
    settings = CASES[case]
    if settings["static"]:
        # a search at rest: no epochs are given, so the searched region is the sky position alone
        return thr.lrt_threshold(settings["psi"], instrument(case), false_rate=false_rate,
                                 half_size=settings["half_size"])
    # a moving search: the epochs and the largest proper motion are given
    return thr.lrt_threshold(settings["psi"], instrument(case), settings["tau"], false_rate=false_rate,
                             mu_max=settings["mu_max"], half_size=settings["half_size"])

In [ ]:
# Lambda_c of every case at false rates of 10 % and 1 %
RATES = (0.10, 0.01)                       # the two nominal false rates: 10 % and 1 %
# LEVEL: a table from a case's name to its two levels (Lambda_c at 10 %, Lambda_c at 1 %), in the order of RATES;
# LEVEL["reference"][1] is the paper's 19.1
LEVEL = {}
for case in NAMES:
    levels_of_case = []
    for false_rate in RATES:
        levels_of_case.append(formula(case, false_rate))
    LEVEL[case] = tuple(levels_of_case)

print(f"{'case':22s} Lambda_c(10 %)  Lambda_c(1 %)")
for case in NAMES:
    print(f"{case:22s} {LEVEL[case][0]:12.2f} {LEVEL[case][1]:13.2f}")

In [ ]:
# the reference is the paper's 19.1; the two cases at rest are the levels that the module's description quotes
# [1]: the level at 1 %
assert round(LEVEL["reference"][1], 2) == 19.13
assert round(LEVEL["at rest, 47 transits"][1], 2) == 17.07 and round(LEVEL["at rest, 95 transits"][1], 2) == 17.31
print("the reference gives 19.13; at rest 17.07 (47 transits) and 17.31 (95 transits)")

In [ ]:
# the curvatures of a case, and the formula's whole curve P(Lambda_max > u), which section 5 draws
def curvatures(case):
    """The curvatures of a case's searched region (thr.search_curvatures).

    Inputs
        case   the case's name, a key of CASES
    Returns the numbers (L_0, ..., L_D), with D the number of dimensions of the region: D = 2 at rest (the sky
    position), D = 4 moving (the position and the proper motion)."""
    settings = CASES[case]
    if settings["static"]:
        return thr.search_curvatures(settings["psi"], instrument(case), half_size=settings["half_size"])
    return thr.search_curvatures(settings["psi"], instrument(case), settings["tau"], mu_max=settings["mu_max"],
                                 half_size=settings["half_size"])


def closed_form(L, u):
    """The closed form of the chance that the strongest candidate of a noise-only search exceeds a level:
    P(max Z > t) = sum over d of L_d rho_d(t), with rho_d(t) = (2 pi)^(-(d+1)/2) He_{d-1}(t) exp(-t^2 / 2).

    Inputs
        L   the curvatures (L_0, ..., L_D) of the searched region, from thr.search_curvatures; L_0 is not used (the
            sum starts at d = 1; L_0 = 0 for these regions)
        u   the level of Lambda = max(Z, 0)^2, a number or an array; the level of Z is t = sqrt(u)
    Returns the chance (the same shape as u). It is an approximation for high levels."""
    t = np.sqrt(u)                         # the level of Z
    # he holds the Hermite polynomials He_0..He_3 of t; he[d - 1] is the He_{d-1} of the term of dimension d (so D is 4 at most)
    he = (np.ones_like(t), t, t ** 2 - 1, t ** 3 - 3 * t)
    # the sum over the dimensions d = 1, ..., D of the searched region; len(L) is D + 1
    chance = 0
    for d in range(1, len(L)):
        chance = chance + L[d] * (2 * np.pi) ** (-(d + 1) / 2) * he[d - 1] * np.exp(-t ** 2 / 2)
    return chance

## 3. The simulated null of each case (Sect. 5.1)

`sim.field_null_max` returns, for each noise-only campaign, the maximum of Z over the searched region. A
search exceeds a level u when max(Z, 0)² > u. Each case has 8 seeds of 200 campaigns.

A search at rest is simulated with `mu_max=0`: the region then has one motion, zero. The function's last step,
which refines the maximum in position and motion, then does nothing (a refined point with a motion is outside
the region), so the maximum is the one of the position grid; the grid is made finer for it, 1.5 mas (6 mas
otherwise).

In [ ]:
# the seeds and the number of campaigns of each seed
SEEDS = np.arange(201, 209)                # the eight seeds: 201, 202, ..., 208
N_PER_SEED = 200                           # the number of campaigns of each seed
STATIC_X_STEP = 1.5                        # the position grid of a search at rest [mas]

In [ ]:
# the first n campaigns of one case and seed (run)
def run(case, seed, n):
    """The strongest candidate of each of the first n noise-only campaigns of one case and seed.

    Inputs
        case   the case's name, a key of CASES
        seed   the seed of the noise, a whole number
        n      the number of campaigns
    Returns n numbers: max Z of each campaign, from sim.field_null_max (the maximum over the searched region of
    the candidate's best-fit flux over its error, in noise-only windows)."""
    settings = CASES[case]
    # the generator of the noise. Its seed is a list of two whole numbers, [the seed, the case's number in NAMES],
    # so every (case, seed) has its own noise
    case_number = NAMES.index(case)
    rng_noise = np.random.default_rng([int(seed), case_number])
    if settings["static"]:
        # a search at rest: the largest motion is zero, and the position grid is the finer one (STATIC_X_STEP)
        return sim.field_null_max(settings["psi"], settings["tau"], n, rng_noise, instrument(case), mu_max=0.0,
                                  half_size=settings["half_size"], x_step=STATIC_X_STEP)
    return sim.field_null_max(settings["psi"], settings["tau"], n, rng_noise, instrument(case),
                              mu_max=settings["mu_max"], half_size=settings["half_size"])

In [ ]:
# the simulated null: recomputed and stored with FULL, otherwise loaded from the store
# max_z: max Z of every campaign, an array over (case, seed, campaign)
if FULL:
    max_z_rows = []                        # one row per case
    for case in NAMES:
        max_z_of_case = []                 # one entry per seed: the N_PER_SEED campaigns of that seed
        for seed in SEEDS:
            max_z_of_case.append(run(case, seed, N_PER_SEED))
        max_z_rows.append(max_z_of_case)
    max_z = np.array(max_z_rows)
    STORE.parent.mkdir(exist_ok=True)
    np.savez(STORE, max_z=max_z, cases=np.array(NAMES), seeds=SEEDS)
    print(f"computed and stored {STORE}")
else:
    with np.load(STORE) as stored:
        max_z = stored["max_z"]
        # the store was made with this notebook's cases and seeds
        assert tuple(stored["cases"]) == NAMES and np.array_equal(stored["seeds"], SEEDS)
    print(f"loaded {STORE}")
assert max_z.shape == (len(NAMES), SEEDS.size, N_PER_SEED)

In [ ]:
# the first campaign of each case recomputed and compared with the store, bit for bit. max_z is (case, seed,
# campaign): [i, 0, :1] keeps case i, the first seed and the first campaign, as an array of one element
for i, case in enumerate(NAMES):           # i: the case's number
    fresh = run(case, SEEDS[0], 1)
    stored_first = max_z[i, 0, :1]
    assert np.array_equal(fresh, stored_first), case
    print(f"{case:22s} campaign 0 of seed {SEEDS[0]}: max Z = {fresh[0]:.6f}, as stored")

## 4. The false rate at the formula's levels, and the simulated levels (Sect. 5.1)

Two comparisons for each case and each nominal rate p (10 % and 1 %):

- the **rate**: the fraction of the simulated campaigns above the formula's level, with its exact 95 %
  interval. The formula is right when the interval holds p.
- the **level**: the simulated level, the (1 − p) quantile of the campaigns, with a bootstrap 95 % interval,
  beside the formula's level.

In [ ]:
# the rate k/n of k searches above a level among n, with its exact interval (rate)
def rate(k, n):
    """The rate of k searches above a level among n, with its exact (Clopper-Pearson) 95 % interval, from the
    beta distribution.

    Inputs
        k   the number of searches above the level
        n   the number of searches
    Returns three numbers
        the rate k / n
        the lower end of its 95 % interval; 0 when k = 0
        the upper end of its 95 % interval; 1 when k = n"""
    if k > 0:
        lower = beta.ppf(0.025, k, n - k + 1)
    else:
        lower = 0.0
    if k < n:
        upper = beta.ppf(0.975, k + 1, n - k)
    else:
        upper = 1.0
    return k / n, lower, upper

In [ ]:
# per case: Lambda_max of every campaign, the rates above the formula's two levels, and the simulated levels
N_BOOTSTRAP = 2000                         # the number of resamples of the bootstrap
# the generator of the bootstrap: one stream for the ten cases, in the order of NAMES. The seed is 0; the reason
# for that value: not stated
rng = np.random.default_rng(0)
# the two quantiles of the simulated levels: 1 - p for each nominal rate p, so 0.90 and 0.99
quantile_of_rate = []
for p in RATES:
    quantile_of_rate.append(1.0 - p)
# R: a table from a case's name to its results, itself a table with five entries; each entry but "lam" has two
# places, for the nominal rates 10 % and 1 %
#     "lam"       Lambda_max = max(Z, 0)^2 of each of its 1600 campaigns
#     "k"         the number of campaigns with Lambda_max above the formula's level
#     "rate"      (k / n, the lower end, the upper end of its exact 95 % interval)
#     "level"     the simulated level: the (1 - p) quantile of Lambda_max
#     "level_ci"  (the lower end, the upper end) of the simulated level's bootstrap 95 % interval
R = {}
for i, case in enumerate(NAMES):           # i: the case's number
    # step 1, Lambda_max of every campaign
    # max_z[i].ravel(): case i, its 8 seeds x 200 campaigns as one vector; np.clip(..., 0, None) is max(Z, 0)
    lam = np.square(np.clip(max_z[i].ravel(), 0.0, None))
    # step 2, the number of campaigns above each of the formula's two levels
    counts = []
    for level in LEVEL[case]:
        counts.append(int((lam > level).sum()))
    # step 3, the simulated levels
    quantiles = []
    for p in RATES:
        quantiles.append(np.quantile(lam, 1.0 - p))
    # step 4, the bootstrap of the simulated levels: as many campaigns as the case has, drawn with replacement,
    # and their two quantiles, N_BOOTSTRAP times; boot has one row per resample and one column per nominal rate
    boot = []
    for _ in range(N_BOOTSTRAP):           # one pass per resample; the counter is not used
        resample = rng.choice(lam, lam.size)
        boot.append(np.quantile(resample, quantile_of_rate))
    boot = np.array(boot)
    # step 5, the rates with their intervals
    rates_of_case = []
    for k in counts:
        rates_of_case.append(rate(k, lam.size))
    # np.percentile(..., axis=0): the 2.5th and the 97.5th percentile over the resamples, a row each
    R[case] = dict(lam=lam, k=counts, rate=rates_of_case, level=quantiles,
                   level_ci=np.percentile(boot, [2.5, 97.5], axis=0).T)    # .T: one (low, high) row per nominal rate

In [ ]:
# the table of the rates: the simulated false rate at the formula's level, for the nominal 10 % and 1 %
print(f"{'case':22s} {'at Lambda_c(10 %)':>26s} {'at Lambda_c(1 %)':>26s}")
for case in NAMES:
    cells = []                             # the two printed cells of the case's row: at 10 %, at 1 %
    for fraction, lower, upper in R[case]["rate"]:
        cells.append(f"{100 * fraction:5.2f} % [{100 * lower:5.2f}, {100 * upper:5.2f}]")
    print(f"{case:22s} {cells[0]:>26s} {cells[1]:>26s}")

In [ ]:
# the table of the levels: the formula's Lambda_c beside the simulated level with its interval
print(f"{'case':22s} {'10 %: formula, simulated':>34s} {'1 %: formula, simulated':>34s}")
for case in NAMES:
    cells = []                             # the two printed cells of the case's row: at 10 %, at 1 %
    for j in range(2):                     # j: the nominal rate's number: 0 for 10 %, 1 for 1 %
        formula_level = LEVEL[case][j]
        simulated_level = R[case]["level"][j]
        lower, upper = R[case]["level_ci"][j]              # the simulated level's bootstrap interval
        cells.append(f"{formula_level:5.2f}, {simulated_level:5.2f} [{lower:5.2f}, {upper:5.2f}]")
    print(f"{case:22s} {cells[0]:>34s} {cells[1]:>34s}")

In [ ]:
# what the simulation supports, asserted: at 1 % every case's interval holds the nominal rate; at 10 % no case
# is above it; all cases together at the 1 % level
# check 1, per case
for case in NAMES:
    (f10, lo10, hi10), (f1, lo1, hi1) = R[case]["rate"]       # the rate and its interval at the two levels
    assert lo1 <= 0.01 <= hi1, case
    assert f10 <= 0.10, case
# check 2, the ten cases together at their 1 % level: the campaigns above it, and all the campaigns
k_all = 0
n_all = 0
for case in NAMES:
    k_all = k_all + R[case]["k"][1]        # [1]: the count at the 1 % level
    n_all = n_all + R[case]["lam"].size
f_all, lo_all, hi_all = rate(k_all, n_all)
print(f"all cases at their 1 % level: {k_all} of {n_all} = {100 * f_all:.2f} % "
      f"[{100 * lo_all:.2f}, {100 * hi_all:.2f}]")
assert lo_all <= 0.01 <= hi_all
# the rate of every case at its 10 % level: ["rate"][0] is the 10 % level's (rate, lower, upper); its [0], the rate
f10s = []
for case in NAMES:
    f10s.append(R[case]["rate"][0][0])
print(f"at their 10 % level the rates are {100 * min(f10s):.1f} to {100 * max(f10s):.1f} %")

**What this shows.** At the level the search uses (1 %), the simulated false rate of every case agrees with
1 % within its interval of 1600 campaigns (about 0.3 to 2 %), and the ten cases together give 1 % to within a
fifth. At the 10 % level the formula is conservative in every case (7 to 9 % simulated): it is an
approximation for high thresholds, and it counts more than the field delivers lower down. The test does not
cover a noise that follows the light or a free primary (both are in `sim_lambda_c.ipynb`, for Figure 4's
case), nor the joint fits of the search itself.

## 5. The figures

Three views of the same comparison.

In [ ]:
# the paper's style, used by the three figures
PAPER_STYLE = {"font.size": 10.0, "xtick.labelsize": 9.0, "ytick.labelsize": 9.0, "axes.linewidth": 0.5,
               "xtick.direction": "in", "ytick.direction": "in", "legend.frameon": False, "legend.fontsize": 8.0}
# [::-1] reverses the order: the first case gets the largest row number
ROWS = np.arange(len(NAMES))[::-1]         # the row of each case, the reference at the top

**The distributions.** One box per case: the simulated Λ_max of its campaigns. The box holds the middle half,
the line in it is the median, and the whiskers end at the 90th and the 99th percentile: the simulated 10 % and
1 % levels. The marks are the formula's levels. The formula is right where each mark is at the end of its
whisker.

In [ ]:
# figure 1: the distribution of the simulated Lambda_max of each case, with the formula's two levels
with plt.rc_context(PAPER_STYLE):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for row, case in zip(ROWS, NAMES):
        # the quartiles of the case's Lambda_max, and its 90th and 99th percentiles (the simulated 10 %, 1 % levels)
        q25, q50, q75, q90, q99 = np.quantile(R[case]["lam"], [0.25, 0.5, 0.75, 0.90, 0.99])
        # the box from the first to the third quartile, the median in it, the whisker from the box to q99
        ax.add_patch(plt.Rectangle((q25, row - 0.28), q75 - q25, 0.56, fc="0.88", ec="k", lw=0.6))
        ax.plot([q50, q50], [row - 0.28, row + 0.28], color="k", lw=1.0)
        ax.plot([q75, q99], [row, row], color="k", lw=0.6)
        for q in (q90, q99):                                               # a tick at each of the two simulated levels
            ax.plot([q, q], [row - 0.14, row + 0.14], color="k", lw=0.6)
        # the formula's two levels; the label goes to the legend once, from the first case (row == ROWS[0])
        if row == ROWS[0]:
            label_10 = "formula, 10 %"
            label_1 = "formula, 1 %"
        else:
            label_10 = None
            label_1 = None
        ax.plot(LEVEL[case][0], row, "v", color="dodgerblue", ms=6, label=label_10)
        ax.plot(LEVEL[case][1], row, "v", color="tomato", ms=6, label=label_1)
    ax.set_yticks(ROWS)
    ax.set_yticklabels(NAMES)
    ax.set(xlabel=r"$\Lambda_{\max}$ of a noise-only campaign", xlim=(4, 24), ylim=(-0.7, len(NAMES) - 0.3))
    ax.legend(loc="lower right")
    fig.tight_layout()
    plt.show()

**The rates.** The simulated false rate at the formula's level (the level the formula gives for that rate), with its 95 % interval, for the nominal 10 %
(left) and 1 % (right). The dotted line is the nominal rate.

In [ ]:
# figure 2: the simulated false rate at the formula's level, per case, against the nominal rate
with plt.rc_context(PAPER_STYLE):
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 4.2), sharey=True)
    for j, (ax, p) in enumerate(zip(axes, RATES)):     # j: the nominal rate's number; ax: its panel; p: the rate
        for row, case in zip(ROWS, NAMES):
            fraction, lower, upper = R[case]["rate"][j]            # the simulated rate and its exact interval
            # xerr takes the distances below and above the point, as two rows
            ax.errorbar(100 * fraction, row, xerr=[[100 * (fraction - lower)], [100 * (upper - fraction)]], fmt="o",
                        color="k", ms=4, lw=0.8, capsize=2)
        ax.axvline(100 * p, color="0.4", lw=0.8, ls=":")           # the nominal rate
        ax.set(xlabel=f"false rate at the {100 * p:.0f} % level  [%]", ylim=(-0.7, len(NAMES) - 0.3))
    axes[0].set_yticks(ROWS)
    axes[0].set_yticklabels(NAMES)
    fig.tight_layout()
    plt.show()

**The survival curves.** For each case, the fraction of the simulated campaigns with Λ_max above u (the steps)
and the formula's curve (the line). The formula is a high-threshold approximation: it is expected to agree at
the right of each panel and to rise above the simulation at the left.

In [ ]:
# figure 3: the simulated survival curve of each case (steps) against the formula's curve (line)
with plt.rc_context(PAPER_STYLE):
    u = np.linspace(6.0, 26.0, 300)        # the levels of Lambda at which the formula's curve is drawn
    fig, axes = plt.subplots(2, 5, figsize=(11.0, 4.6), sharex=True, sharey=True)
    for ax, case in zip(axes.ravel(), NAMES):                               # ravel: the 2 x 5 panels as one row of 10
        lam = np.sort(R[case]["lam"])      # Lambda_max of the campaigns, in increasing order
        # the fraction of the campaigns above each sorted Lambda_max: 1 - (its rank) / (the number of campaigns)
        surv = 1.0 - np.arange(1, lam.size + 1) / lam.size
        ax.step(lam, surv, where="post", color="k", lw=0.9)
        ax.plot(u, closed_form(curvatures(case), u), color="tomato", lw=1.1)
        ax.axhline(0.01, color="0.5", lw=0.5, ls=":")                       # the two nominal rates
        ax.axhline(0.10, color="0.5", lw=0.5, ls=":")
        ax.set_title(case, fontsize=8.5)
        ax.set(yscale="log", xlim=(u[0], u[-1]), ylim=(5e-4, 1.5))
    for ax in axes[-1]:                                                     # axes[-1]: the bottom row
        ax.set_xlabel(r"$u$")
    for ax in axes[:, 0]:                                                   # axes[:, 0]: the left column
        ax.set_ylabel(r"$P(\Lambda_{\max} > u)$")
    fig.tight_layout()
    plt.show()